# Инференс на 402 фотографиях фасадов зданий

In [ ]:
import numpy as np
import pandas as pd
import os
import torch
from scipy.stats import skew as scipy_skew
from torch.nn import functional as F
from pathlib import Path
import cv2
from tqdm import tqdm

## Задаем пути, классы и константы

In [ ]:
classes = ['coating_deterioration', 'masonry_degradation',
           'moisture_bio_damage', 'vandalism']


softmax_threshold = 0.25
epsilon = 0.01
tile_size = 800
stride = 700


color_map = np.array([
    [0,   0,   0  ],   # 0 background
    [54,  21,  217],   # 1 coating_deterioration
    [42,  125, 209],   # 2 masonry_degradation
    [38,  221, 98 ],   # 3 moisture_bio_damage
    [222, 28,  123],   # 4 vandalism
], dtype=np.uint8)

num_classes = 5


image_dir = '/kaggle/input/datasets/neuromant/402-buildings-v2/inference-402-imgs-v2' 
output_csv = '/kaggle/working/building_features_402-32d.csv'
model_path = '/kaggle/input/models/neuromant/segformer-b2-505-imgs-4-cls-ff/pytorch/default/1/final_fit_505_4_classes_3200'
masks_dir  = '/kaggle/working/masks_inference-402'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Масштабирование изображений
def get_scale_factor(w, h):
    if w <= 2000:   return 1.41          # Яндекс.Панорамы
    elif w >= 5000: return 800 / 1200    # Зеркалка
    else:           return 1.0           # Телефон

def get_starts(size, tile, stride):
    """Последний тайл добавляется только если вносит
    не менее tile//2 новых пикселей"""
    if size <= tile:
        return [0]
    starts = list(range(0, size - tile, stride))
    last = size - tile
    if (last - starts[-1]) >= tile // 2:
        starts.append(last)
    return starts

# Генерируем по 8 статисчтических показателей на каждый класс
def compute_features(ratios_per_class):
    features = {}
    for cls, ratios in ratios_per_class.items():
        r = np.array(ratios, dtype=np.float32)
        N = len(r)

        mean_val = float(np.mean(r))
        max_val = float(np.max(r))
        prevalence = float(np.mean(r > epsilon))
        std_val = float(np.std(r))
        q75_val = float(np.percentile(r, 75))
        skewness = float(scipy_skew(r)) if (N >= 3 and np.std(r) > 0) else 0.0
        concentration    = (mean_val / max_val) if max_val >= epsilon else 0.0
        severity_coverage = max_val * prevalence

        for stat, val in zip(
            ['mean', 'max', 'prevalence', 'std', 'q75',
             'skewness', 'concentration', 'severity_coverage'],
            [mean_val, max_val, prevalence, std_val, q75_val,
             skewness, concentration, severity_coverage]
        ):
            features[f'{cls}_{stat}'] = val
    return features

In [ ]:
# Инференс одного здания
def infer_building(img_path, model, processor, masks_dir):
    '''
    функция берет одну фотографию фасада, прогоняет ее через модель
    сегментации, сохраняет сегментационные RGB маски в папку и извлекает полезные признаки о повреждениях, сохраняя их в .csv файл
    '''
    # Считываем изображения
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return None

    h0, w0 = img_bgr.shape[:2]
    # Вычисляем, нужно ли нам изменить размер фото
    scale = get_scale_factor(w0, h0)
    if scale != 1.0:
        # Выбираем метод интерполяции
        interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
        # Изменяем размер фото
        img_bgr = cv2.resize(img_bgr,
                             (int(w0 * scale), int(h0 * scale)),
                             interpolation=interp)
    # Переводим из BGR в RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    # Создаем массивы для накопления вероятностей классов для каждого пикселя
    # и для подсчета, сколько раз каждый пиксель был перекрыт тайлами
    prob_sum = np.zeros((num_classes, h, w), dtype=np.float32)
    count    = np.zeros((h, w), dtype=np.float32)
    
    # Определяем начальные координаты для нарезки фото на тайлы
    y_starts = get_starts(h, tile_size, stride)
    x_starts = get_starts(w, tile_size, stride)
    # Создаем словарь, где будем хранить доли каждого класса повреждений
    # для каждого тайла
    ratios_per_class = {cls: [] for cls in classes}

    with torch.no_grad():
        # Проходимся по всем тайлам
        for y in y_starts:
            for x in x_starts:
                # Вырезаем текущий тайл из изображения
                tile = img_rgb[y:y+tile_size, x:x+tile_size]
                inputs = processor(images=tile, return_tensors='pt').to(device)

                logits = model(**inputs).logits
                probs  = F.softmax(
                    F.interpolate(logits,
                        size=(tile_size, tile_size),
                        mode='bilinear', align_corners=False),
                    dim=1
                )[0].cpu().numpy()                     

                total = tile_size * tile_size
                # Определяем фактические границы тайла, учитывая края изображения.
                y_end = min(y + tile_size, h) 
                x_end = min(x + tile_size, w)
                # Вычисляем фактическую высоту и ширину текущего участка тайла.
                ph, pw = y_end - y, x_end - x
                # Накапливаем вероятности для каждого пикселя.
                prob_sum[:, y:y_end, x:x_end] += probs[:, :ph, :pw]
                # Увеличиваем счетчик для каждого пикселя, показывая, сколько раз он был покрыт тайлом.
                count[      y:y_end, x:x_end] += 1

                # Проходимся по каждому классу повреждений
                for cls_idx, cls_name in enumerate(classes, start=1):
                    # Считаем долю пикселей, которые модель отнесла к текущему классу с уверенностью выше порога 
                    ratio = float(np.sum(probs[cls_idx] > softmax_threshold) / total)
                    # Записываем эту долю в наш словарь
                    ratios_per_class[cls_name].append(ratio)

    # Усредняем накопленные вероятности для каждого пикселя.
    prob_avg   = prob_sum / np.maximum(count, 1)[np.newaxis]
    # Для каждого пикселя выбираем класс с наибольшей средней вероятностью
    # Это и будет наша итоговая маска с индексами классов
    mask_index = prob_avg.argmax(axis=0).astype(np.uint8)
    # Преобразуем маску с индексами классов в цветную маску
    mask_color = color_map[mask_index] 
    # Получаем базовое имя файла изображения
    stem = img_path.stem
    # Сохраняем маску с индексами классов
    cv2.imwrite(os.path.join(masks_dir, f"{stem}_mask.png"), mask_index)
    # Сохраняем цветную маску
    cv2.imwrite(os.path.join(masks_dir, f"{stem}_color.png"),
                cv2.cvtColor(mask_color, cv2.COLOR_RGB2BGR))          
    # агрегируем полученные доли  в общие признаки для всего здания
    features = compute_features(ratios_per_class)
    features['num_tiles'] = len(y_starts) * len(x_starts)
    
    return features

In [ ]:
def run_inference(image_dir, model, processor):
    '''
    Функция прогоняет все здания через модель, вычисляет статистические показатели по сгементационным маскам и сохраняет их в .csv файл
    '''
    # пути ко всем фото
    paths = []
    for ext in ['*.jpg', '*.JPG', '*.png', '*.PNG']:
        # Добавляем найденные пути в наш список
        paths.extend(list(Path(image_dir).glob(ext)))
    print(f"Зданий для инференса: {len(paths)}")

    model.eval()
    model.to(device)
    # Создаем список, куда будем собирать результаты по каждому зданию
    records = []
    for img_path in tqdm(paths, desc='Инференс'):
        # получаем признаки для текущего здания
        feats = infer_building(img_path, model, processor)
        # Добавляем имя файла для идентификации здания
        feats['building_name'] = img_path.stem
        records.append(feats)
    # Собираем результаты
    df = pd.DataFrame(records)

    feat_cols = [f'{cls}_{stat}'
                 for cls in classes
                 for stat in ['mean', 'max', 'prevalence', 'std', 'q75',
                              'skewness', 'concentration', 'severity_coverage']]
    df = df[['building_name', 'num_tiles'] + feat_cols]   
  
    # Cохраняем
    df.to_csv(output_csv, index=False)
    print(f"Сохранено: {output_csv}")
    return df

## Делаем инференс

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(model_path)
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b2")

model.to(device)
model.eval()
features_df = run_inference(image_dir, model, processor)
features_df.head()